In [3]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------

In [4]:
import os
import re
from io import BytesIO
from urllib.parse import urlparse

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================
# Set this to either:
#  - a local file path, e.g. r"C:\Users\you\Downloads\annex-private-motor-insurance-report-7.xlsx"
#  - or a direct URL to the workbook
SOURCE = r"annex-private-motor-insurance-report-7.xlsx"

# These are the sheet names you want conceptually.
# The code will map them to the workbook's actual tab names
# (e.g. "Figure 14_19" -> "Figure14_19", "Table11" -> "Table 11")
TARGET_SHEETS = [
    "PremData",
    "UltData",
    "Figure 14_19",
    "Figure 23",
    "Table9",
    "Table10",
    "Table11",
    "Table12_13",
]

OUTPUT_DIR = "tidy_outputs"


# ============================================================
# SMALL UTILITIES
# ============================================================
def clean_path(path):
    return str(path).strip().strip('"').strip("'")


def is_url(value):
    parsed = urlparse(str(value))
    return parsed.scheme in ("http", "https")


def normalize_text(x):
    if pd.isna(x):
        return None
    s = str(x).replace("\xa0", " ").strip()
    return s if s else None


def is_blank(x):
    return normalize_text(x) is None


def year_to_str(x):
    if pd.isna(x):
        return None
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, float):
        if abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        return str(x)
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s


def is_year_value(x):
    if pd.isna(x):
        return False

    if isinstance(x, (int, np.integer)):
        y = int(x)
        return 1900 <= y <= 2100

    if isinstance(x, float):
        return abs(x - round(x)) < 1e-9 and 1900 <= int(round(x)) <= 2100

    s = str(x).strip()
    return bool(re.fullmatch(r"(19|20)\d{2}(?:\.0+)?", s))


def clean_numeric_value(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    s = str(x).replace(",", "").replace("€", "").replace("%", "").strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan

    try:
        return float(s)
    except Exception:
        return np.nan


def clean_df(df):
    """Drop completely empty rows and columns."""
    return df.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)


def clean_rows_only(df):
    """Drop completely empty rows but keep columns (used for UltData spacer column)."""
    return df.dropna(how="all").reset_index(drop=True)


def format_filename(text):
    text = str(text).strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_")


def canonical_sheet_name(name):
    """Normalize sheet names so that:
       - 'Figure 14_19' == 'Figure14_19'
       - 'Table11' == 'Table 11'
    """
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def resolve_sheet_name(sheet_names, requested_name):
    wanted = canonical_sheet_name(requested_name)
    mapping = {canonical_sheet_name(s): s for s in sheet_names}
    if wanted in mapping:
        return mapping[wanted]
    raise KeyError(f"Sheet not found in workbook: {requested_name}")


def looks_like_footer(text):
    s = (normalize_text(text) or "").lower()
    footer_terms = [
        "coverage",
        "central bank of ireland - unrestricted",
        "source",
        "note",
        "notes",
        "* to ensure",
    ]
    return any(term in s for term in footer_terms)


def find_header_row_by_required_cols(df, required_sets, max_scan_rows=10):
    """
    Find a header row whose normalized values contain all labels in one of the required sets.
    """
    for i in range(min(len(df), max_scan_rows)):
        vals = [normalize_text(v) for v in df.iloc[i].tolist()]
        lowered = {v.lower() for v in vals if v}
        for req in required_sets:
            if req.issubset(lowered):
                return i
    return None


# ============================================================
# LOAD WORKBOOK
# ============================================================
def load_workbook(source):
    source = clean_path(source)

    if is_url(source):
        import requests

        response = requests.get(source, timeout=60)
        response.raise_for_status()
        return pd.read_excel(BytesIO(response.content), sheet_name=None, header=None, engine="openpyxl")

    return pd.read_excel(source, sheet_name=None, header=None, engine="openpyxl")


# ============================================================
# PARSERS
# ============================================================
def parse_premdata(df):
    """
    PremData is already a long table with:
      Year | YearH | AccidentQuarter | Measure | CoverType | Value
    """
    df = clean_df(df)

    hdr = find_header_row_by_required_cols(
        df,
        [set(["year", "yearh", "accidentquarter", "measure", "covertype", "value"])]
    )
    if hdr is None:
        raise ValueError("Could not find PremData header row")

    data = df.iloc[hdr + 1:].copy().reset_index(drop=True)
    data.columns = [normalize_text(v) for v in df.iloc[hdr].tolist()]
    data = clean_df(data)

    data = data[~data.iloc[:, 0].apply(looks_like_footer)].copy()

    data = data.rename(columns={
        "Year": "year",
        "YearH": "year_half",
        "AccidentQuarter": "accident_quarter",
        "Measure": "measure",
        "CoverType": "cover_type",
        "Value": "value",
    })

    data["year"] = data["year"].apply(year_to_str)
    data["accident_quarter"] = data["accident_quarter"].apply(lambda x: year_to_str(x) if pd.notna(x) else None)
    data["value"] = data["value"].apply(clean_numeric_value)

    data = data[data["value"].notna()].reset_index(drop=True)

    return data[["year", "year_half", "accident_quarter", "measure", "cover_type", "value"]]


def parse_ultdata(df):
    """
    UltData has two side-by-side tables:

      LEFT  = Year | AccidentQuarter | Measure | Claim Type | Value
      RIGHT = Year | AccidentQuarter | Measure | CoverType  | Value

    There is a blank spacer column between them, so we must NOT drop all-empty columns.
    """
    df = clean_rows_only(df)

    hdr = None
    for i in range(min(len(df), 10)):
        vals = [normalize_text(v) for v in df.iloc[i].tolist()]
        vals_lower = [v.lower() if v else None for v in vals]

        if (
            "year" in vals_lower
            and "accidentquarter" in vals_lower
            and "measure" in vals_lower
            and "claim type" in vals_lower
            and "covertype" in vals_lower
            and "value" in vals_lower
        ):
            hdr = i
            break

    if hdr is None:
        raise ValueError("Could not find UltData dual header row")

    header_vals = [normalize_text(v) for v in df.iloc[hdr].tolist()]

    left_year_idx = header_vals.index("Year")
    right_year_idx = len(header_vals) - 1 - header_vals[::-1].index("Year")

    left_cols = list(range(left_year_idx, left_year_idx + 5))
    right_cols = list(range(right_year_idx, right_year_idx + 5))

    left_header = [normalize_text(df.iat[hdr, c]) for c in left_cols]
    right_header = [normalize_text(df.iat[hdr, c]) for c in right_cols]

    left = df.iloc[hdr + 1:, left_cols].copy().reset_index(drop=True)
    right = df.iloc[hdr + 1:, right_cols].copy().reset_index(drop=True)

    left.columns = left_header
    right.columns = right_header

    left = clean_df(left)
    right = clean_df(right)

    left = left[~left.iloc[:, 0].apply(looks_like_footer)].copy()
    right = right[~right.iloc[:, 0].apply(looks_like_footer)].copy()

    left = left.rename(columns={
        "Year": "year",
        "AccidentQuarter": "accident_quarter",
        "Measure": "measure",
        "Claim Type": "dimension_value",
        "Value": "value",
    })
    left["dimension_type"] = "claim_type"

    right = right.rename(columns={
        "Year": "year",
        "AccidentQuarter": "accident_quarter",
        "Measure": "measure",
        "CoverType": "dimension_value",
        "Value": "value",
    })
    right["dimension_type"] = "cover_type"

    for block in (left, right):
        block["year"] = block["year"].apply(year_to_str)
        block["accident_quarter"] = block["accident_quarter"].apply(lambda x: year_to_str(x) if pd.notna(x) else None)
        block["value"] = block["value"].apply(clean_numeric_value)

    out = pd.concat(
        [
            left[["year", "accident_quarter", "measure", "dimension_type", "dimension_value", "value"]],
            right[["year", "accident_quarter", "measure", "dimension_type", "dimension_value", "value"]],
        ],
        ignore_index=True,
    )

    out = out[out["value"].notna()].reset_index(drop=True)
    return out


def parse_generic_stacked_year_sheet(df, sheet_name):
    """
    Generic parser for sheets built from stacked blocks with year headers across columns.

    Output columns:
      sheet, table_title, section, label, label_id, year, value
    """
    df = clean_df(df)

    out = []
    current_title = None
    current_section = None
    i = 0

    while i < len(df):
        row = df.iloc[i].tolist()
        nonblank_positions = [idx for idx, v in enumerate(row) if not is_blank(v)]

        if not nonblank_positions:
            i += 1
            continue

        year_cols = [idx for idx, v in enumerate(row) if is_year_value(v)]

        if len(year_cols) >= 3:
            years = [year_to_str(row[c]) for c in year_cols]
            i += 1
            current_section = None

            while i < len(df):
                r = df.iloc[i].tolist()
                nonblank_positions_r = [idx for idx, v in enumerate(r) if not is_blank(v)]

                if not nonblank_positions_r:
                    i += 1
                    continue

                next_year_cols = [idx for idx, v in enumerate(r) if is_year_value(v)]
                if len(next_year_cols) >= 3:
                    break

                first = normalize_text(r[0])

                if first and looks_like_footer(first):
                    break

                # A single-cell row inside a block is usually a section row like Income / Expenses
                # or a new Accompanies... title row.
                if len(nonblank_positions_r) == 1 and first:
                    if first.lower().startswith("accompanies "):
                        current_title = first
                        current_section = None
                    else:
                        current_section = first
                    i += 1
                    continue

                label = normalize_text(r[0]) if len(r) > 0 else None
                label_id = normalize_text(r[1]) if len(r) > 1 else None

                if label is None:
                    i += 1
                    continue

                for pos, yr in zip(year_cols, years):
                    val = clean_numeric_value(r[pos] if pos < len(r) else np.nan)
                    if pd.notna(val):
                        out.append({
                            "sheet": sheet_name,
                            "table_title": current_title,
                            "section": current_section,
                            "label": label,
                            "label_id": label_id,
                            "year": yr,
                            "value": val,
                        })

                i += 1

            continue

        # Single-cell row before a year block = block title
        if len(nonblank_positions) == 1:
            first = normalize_text(row[nonblank_positions[0]])
            if first:
                current_title = first
                current_section = None

        i += 1

    return pd.DataFrame(out)


def parse_figure23(df):
    """
    Figure23 is almost a normal year-across-columns block, but it has one awkward orphan row:
    the 2024 value for 'Direct and Related Distribution' is separated from the measure row.
    """
    df = clean_df(df)

    header_row = None
    for i in range(min(10, len(df))):
        year_cols = [idx for idx, v in enumerate(df.iloc[i].tolist()) if is_year_value(v)]
        if len(year_cols) >= 3:
            header_row = i
            break

    if header_row is None:
        raise ValueError("Could not find Figure23 year header row")

    header = df.iloc[header_row].tolist()
    year_cols = [idx for idx, v in enumerate(header) if is_year_value(v)]

    out = []
    pending_measure = None
    title = normalize_text(df.iloc[1, 0]) if len(df) > 1 else "Figure23"

    for i in range(header_row + 1, len(df)):
        row = df.iloc[i].tolist()
        first = normalize_text(row[0])

        if first and looks_like_footer(first):
            break

        if first and first.lower().startswith("* to ensure"):
            # The next row contains the orphan 2024 value for Direct and Related Distribution
            pending_measure = "Direct and Related Distribution"
            continue

        if first:
            pending_measure = first

            for idx in year_cols:
                val = clean_numeric_value(row[idx])
                if pd.notna(val):
                    out.append({
                        "sheet": "Figure23",
                        "table_title": title,
                        "measure": first,
                        "year": year_to_str(header[idx]),
                        "value": val,
                    })

        elif pending_measure == "Direct and Related Distribution":
            # Workbook-specific orphan value row
            for idx, cell in enumerate(row):
                val = clean_numeric_value(cell)
                if pd.notna(val):
                    if idx in year_cols:
                        yr = year_to_str(header[idx])
                    else:
                        # Fallback for the orphan value row in this workbook
                        yr = "2024"

                    out.append({
                        "sheet": "Figure23",
                        "table_title": title,
                        "measure": pending_measure,
                        "year": yr,
                        "value": val,
                    })

    out = pd.DataFrame(out)
    return out.sort_values(["measure", "year"]).reset_index(drop=True)


def parse_table9_or_10(df, sheet_name):
    """
    Table9 / Table10 are simple yearly tables.
    Output:
      sheet, year, measure, value
    """
    df = clean_df(df)

    header_row = None
    for i in range(min(10, len(df))):
        first = normalize_text(df.iat[i, 0]) if df.shape[1] else None
        second = normalize_text(df.iat[i, 1]) if df.shape[1] > 1 else None
        if first and "year" in first.lower() and second:
            header_row = i
            break

    if header_row is None:
        raise ValueError(f"Could not find header row for {sheet_name}")

    header = [normalize_text(v) for v in df.iloc[header_row].tolist()]
    data = df.iloc[header_row + 1:].copy().reset_index(drop=True)
    data.columns = header
    data = clean_df(data)

    id_col = header[0]
    data = data[~data[id_col].apply(looks_like_footer)].reset_index(drop=True)

    value_cols = [c for c in data.columns if c != id_col]
    for c in value_cols:
        data[c] = data[c].apply(clean_numeric_value)

    out = data.melt(id_vars=[id_col], value_vars=value_cols, var_name="measure", value_name="value")
    out = out[out["value"].notna()].reset_index(drop=True)
    out = out.rename(columns={id_col: "year"})
    out["year"] = out["year"].apply(year_to_str)
    out.insert(0, "sheet", sheet_name)

    return out


def parse_table11(df):
    """
    Table11 needs special handling because the header is:

      Settled Year | Damage | Injury | Total | Damage | Injury | Total

    where:
      first triplet  = claimant numbers
      second triplet = settled costs
    """
    df = clean_df(df)

    header_row = None
    for i in range(min(10, len(df))):
        first = normalize_text(df.iat[i, 0]) if df.shape[1] else None
        if first and "settled year" in first.lower():
            header_row = i
            break

    if header_row is None:
        raise ValueError("Could not find Table11 header row")

    data = df.iloc[header_row + 1:].copy().reset_index(drop=True)
    data.columns = [
        "settled_year",
        "claimant_numbers_damage",
        "claimant_numbers_injury",
        "claimant_numbers_total",
        "settled_cost_damage",
        "settled_cost_injury",
        "settled_cost_total",
    ]
    data = clean_df(data)

    data = data[~data["settled_year"].apply(looks_like_footer)].reset_index(drop=True)

    for c in data.columns[1:]:
        data[c] = data[c].apply(clean_numeric_value)

    mapping = [
        ("claimant_numbers", "damage", "claimant_numbers_damage"),
        ("claimant_numbers", "injury", "claimant_numbers_injury"),
        ("claimant_numbers", "total", "claimant_numbers_total"),
        ("settled_cost", "damage", "settled_cost_damage"),
        ("settled_cost", "injury", "settled_cost_injury"),
        ("settled_cost", "total", "settled_cost_total"),
    ]

    out = []
    for _, r in data.iterrows():
        yr = year_to_str(r["settled_year"])
        for metric_group, claim_type, col in mapping:
            if pd.notna(r[col]):
                out.append({
                    "sheet": "Table11",
                    "settled_year": yr,
                    "metric_group": metric_group,
                    "claim_type": claim_type,
                    "value": float(r[col]),
                })

    return pd.DataFrame(out)


def parse_table12_13(df):
    """
    Table12_13 contains TWO stacked tables on one worksheet:
      - Table 12
      - Table 13

    Each one has:
      Years row
      section rows like 'Settled Claimant Numbers' / 'Settled Claim Costs'
      category rows beneath
    """
    df = clean_df(df)
    out = []
    i = 0

    while i < len(df):
        first = normalize_text(df.iat[i, 0]) if df.shape[1] else None

        if first and first.lower().startswith("accompanies "):
            title = first
            j = i + 1

            if j < len(df):
                row = df.iloc[j].tolist()
                year_cols = [idx for idx, v in enumerate(row) if is_year_value(v)]

                if len(year_cols) >= 3:
                    years = [year_to_str(row[c]) for c in year_cols]

                    m = re.search(r"table\s+(\d+)", title, flags=re.I)
                    subtable = f"Table{m.group(1)}" if m else "Table12_13"

                    section = None
                    k = j + 1

                    while k < len(df):
                        rowk = df.iloc[k].tolist()
                        firstk = normalize_text(rowk[0])

                        if firstk and firstk.lower().startswith("accompanies "):
                            break

                        if firstk and looks_like_footer(firstk):
                            break

                        if all(is_blank(v) for v in rowk):
                            k += 1
                            continue

                        # Section row
                        if firstk and all(is_blank(v) for v in rowk[1:]):
                            section = firstk
                            k += 1
                            continue

                        category = firstk
                        if category:
                            for pos, yr in zip(year_cols, years):
                                val = clean_numeric_value(rowk[pos])
                                if pd.notna(val):
                                    out.append({
                                        "sheet": "Table12_13",
                                        "subtable": subtable,
                                        "table_title": title,
                                        "section": section,
                                        "category": category,
                                        "year": yr,
                                        "value": val,
                                    })

                        k += 1

                    i = k
                    continue

        i += 1

    return pd.DataFrame(out)


# ============================================================
# SHEET ROUTER
# ============================================================
def tidy_sheet(requested_name, df):
    key = canonical_sheet_name(requested_name)

    if key == "premdata":
        return parse_premdata(df)

    if key == "ultdata":
        return parse_ultdata(df)

    if key == "figure1419":
        return parse_generic_stacked_year_sheet(df, "Figure14_19")

    if key == "figure23":
        return parse_figure23(df)

    if key == "table9":
        return parse_table9_or_10(df, "Table9")

    if key == "table10":
        return parse_table9_or_10(df, "Table10")

    if key == "table11":
        return parse_table11(df)

    if key == "table1213":
        return parse_table12_13(df)

    raise KeyError(f"Unsupported target sheet: {requested_name}")


# ============================================================
# MAIN
# ============================================================
def main():
    df_dict = load_workbook(SOURCE)
    workbook_sheet_names = list(df_dict.keys())

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    resolved = []
    for requested in TARGET_SHEETS:
        actual = resolve_sheet_name(workbook_sheet_names, requested)
        resolved.append((requested, actual))

    print("Resolved sheets:")
    for requested, actual in resolved:
        print(f"  {requested}  ->  {actual}")

    for requested, actual in resolved:
        print(f"\nProcessing: {requested} (workbook tab: {actual})")

        try:
            tidy_df = tidy_sheet(requested, df_dict[actual])

            if tidy_df is None or tidy_df.empty:
                print("  Skipped (empty after parsing)")
                continue

            out_name = f"{format_filename(requested)}.csv"
            out_path = os.path.join(OUTPUT_DIR, out_name)
            tidy_df.to_csv(out_path, index=False)

            print(f"  Saved: {out_path}")
            print(f"  Rows: {len(tidy_df)}")
            print(f"  Columns: {list(tidy_df.columns)}")

        except Exception as e:
            print(f"  ERROR on {requested}: {e}")

    print("\nDone.")


if __name__ == "__main__":
    main()

Resolved sheets:
  PremData  ->  PremData
  UltData  ->  UltData
  Figure 14_19  ->  Figure14_19
  Figure 23  ->  Figure23
  Table9  ->  Table9
  Table10  ->  Table10
  Table11  ->  Table 11
  Table12_13  ->  Table12_13

Processing: PremData (workbook tab: PremData)
  Saved: tidy_outputs\PremData.csv
  Rows: 240
  Columns: ['year', 'year_half', 'accident_quarter', 'measure', 'cover_type', 'value']

Processing: UltData (workbook tab: UltData)
  Saved: tidy_outputs\UltData.csv
  Rows: 1080
  Columns: ['year', 'accident_quarter', 'measure', 'dimension_type', 'dimension_value', 'value']

Processing: Figure 14_19 (workbook tab: Figure14_19)
  Saved: tidy_outputs\Figure_14_19.csv
  Rows: 404
  Columns: ['sheet', 'table_title', 'section', 'label', 'label_id', 'year', 'value']

Processing: Figure 23 (workbook tab: Figure23)
  Saved: tidy_outputs\Figure_23.csv
  Rows: 36
  Columns: ['sheet', 'table_title', 'measure', 'year', 'value']

Processing: Table9 (workbook tab: Table9)
  Saved: tidy_outp